# Appendix: Zhao-style and kernel-basis experiments

This notebook contains the Kang--Schafer balance path and the kernel-basis experiment motivated by tailored loss estimation.


In [ ]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.special import expit

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists():
    parent = REPO_ROOT.parent
    if parent == REPO_ROOT:
        raise RuntimeError("Run this notebook from inside the genriesz repository.")
    REPO_ROOT = parent

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from genriesz import grr_ate, grr_att
from genriesz.basis import BaseBasis, TreatmentInteractionBasis
from genriesz.experiments import (
    COMPATIBLE_LOSSES,
    ESTIMANDS,
    ESTIMATORS_ALL,
    RANDOM_SEED,
    TREATMENT_INDEX,
    CoverageDiagnosticBasis,
    SelectedColumnsBasis,
    fit_grr_love_plot_data,
    fit_matching_ate,
    fit_one_grr,
    fit_one_grr_with_basis,
    fit_one_incompatible,
    fit_one_plugin_logistic,
    generator_shift_for_estimand,
    load_ihdp_replication,
    load_lalonde,
    make_basis,
    make_compatible_generator,
    make_coverage_diagnostic_data,
    make_dimension_data,
    make_kang_schafer_data,
    make_kernel_gp_data,
    make_score_guided_data,
    make_simulation_data,
    result_to_rows,
    summarize_estimates,
    true_theta,
)

DATA_DIR = REPO_ROOT / "notebooks" / "experiments" / "data"
TABLE_CONFIG = {"float_format": "{:.4f}", "max_rows": 200}
PLOT_CONFIG = {
    "figure_size": (9.0, 5.2),
    "figure_size_wide": (12.0, 5.2),
    "title_fontsize": 14,
    "axis_fontsize": 12,
    "tick_fontsize": 10,
    "legend_fontsize": 10,
    "line_width": 2.0,
    "marker_size": 5,
    "box_width": 0.70,
    "grid_alpha": 0.30,
    "dpi": 140,
    "squared_error_y_scale": "log",
    "squared_error_floor": 1e-12,
}
METHOD_LABELS = {
    "ra": "RA",
    "rw": "RW (IPW)",
    "arw": "ARW (AIPW)",
    "tmle": "TMLE",
    "SQ": "SQ-Riesz",
    "UKL": "UKL-Riesz",
    "BKL": "BKL-Riesz",
    "BP(0.5)": "BP-Riesz (omega = 0.5)",
    "rkhs": "RKHS",
    "polynomial": "Polynomial",
    "rf": "Random forest leaves",
    "rff": "Random Fourier features",
    "matching": "Nearest-neighbor matching",
}
METHOD_COLORS = {
    "SQ": "#4C78A8",
    "UKL": "#F58518",
    "BKL": "#54A24B",
    "BP(0.5)": "#B279A2",
    "rkhs": "#4C78A8",
    "polynomial": "#F58518",
    "rf": "#54A24B",
    "rff": "#E45756",
    "matching": "#72B7B2",
}
DISPLAY_LABELS = {
    "ra": "RA",
    "rw": "RW",
    "arw": "ARW",
    "tmle": "TMLE",
    "rkhs": "RKHS",
    "polynomial": "Polynomial",
    "rf": "Random Forest",
    "rff": "Random Fourier Features",
    "matching": "Nearest-Neighbor Matching",
    "regressor": "Regressor",
    "covariate": "Covariate",
}
LABEL_COLUMNS = ("estimator", "basis", "basis_mode", "loss", "loss_link_pair")
pd.options.display.max_rows = TABLE_CONFIG["max_rows"]


def label_of(value):
    """Return the display label for a stored result key."""

    return DISPLAY_LABELS.get(str(value), str(value))


def prettify_method(text):
    """Replace stored result keys inside a composite display label."""

    out = str(text)
    for key, value in DISPLAY_LABELS.items():
        out = re.sub(r"(?<![A-Za-z0-9_])" + re.escape(key) + r"(?![A-Za-z0-9_])", value, out)
    return out


def prettify_labels(frame):
    """Return a copy with known result-key columns formatted for display."""

    out = frame.copy()
    for column in LABEL_COLUMNS:
        if column in out.columns:
            out[column] = out[column].map(label_of)
    return out


def display_table(frame, *, caption=None, digits=4):
    """Display a rounded table without changing the stored results."""

    table = prettify_labels(frame)
    numeric_columns = table.select_dtypes(include=[np.number]).columns
    table[numeric_columns] = table[numeric_columns].round(digits)
    styler = table.style.format(precision=digits)
    if caption is not None:
        styler = styler.set_caption(caption)
    display(styler)


In [ ]:
N_REPLICATIONS = 100
SAMPLE_SIZE_ZHAO = 300
SAMPLE_SIZE_KERNEL = 600
RIEZ_LAMBDA = 3e-1
FOLDS = 5
ACTIVE_FEATURE_COUNTS = list(range(1, 9))
KERNEL_BASIS_KINDS = ["rkhs", "polynomial", "rff"]
KERNEL_OUTCOMES = ["polynomial", "sinusoidal"]

## A. Zhao/Kang--Schafer balance path

Features enter progressively. The notebook reports maximum weighted SMD for each number of active features and each loss.

In [ ]:
rows = []
for rep in range(N_REPLICATIONS):
    data = make_kang_schafer_data(n=SAMPLE_SIZE_ZHAO, seed=800000 + 1009 * rep)
    for k in ACTIVE_FEATURE_COUNTS:
        for loss_spec in COMPATIBLE_LOSSES:
            for estimand in ESTIMANDS:
                rows.extend(
                    fit_one_grr_with_basis(
                        data,
                        estimand=estimand,
                        loss_spec=loss_spec,
                        representer_basis=SelectedColumnsBasis(n_active=k),
                        cross_fit=True,
                        lam=RIEZ_LAMBDA,
                        folds=FOLDS,
                        estimators=("rw",),
                        max_iter=500,
                        random_state=rep,
                        label_info={
                            "replication": rep,
                            "estimand": estimand,
                            "loss": loss_spec["label"],
                            "n_active_features": k,
                        },
                    )
                )
zhao_path_results = pd.DataFrame(rows)
zhao_path_summary = summarize_estimates(
    zhao_path_results,
    ["estimand", "loss", "n_active_features", "estimator"],
)

for estimand_name in ESTIMANDS:
    table_df = zhao_path_summary[zhao_path_summary["estimand"] == estimand_name].copy()
    if table_df.empty:
        print(f"No rows for {estimand_name}.")
        continue
    table_df = table_df.drop(columns=["estimand"])
    table_df = table_df.sort_values(["loss", "n_active_features", "estimator"])
    balance_columns = [
        column
        for column in [
            "loss",
            "n_active_features",
            "estimator",
            "max_abs_smd_weighted_mean",
            "alpha_abs_p95_mean",
            "alpha_abs_max_mean",
            "ess_treated_mean",
            "ess_control_mean",
            "coverage",
            "riesz_clip_binding_rate_max_mean",
            "riesz_clip_binding_rate_max_max",
            "riesz_modifies_estimand_mean",
        ]
        if column in table_df.columns
    ]
    display_table(
        table_df[balance_columns],
        caption=f"Zhao/Kang-Schafer balance-path diagnostics: {estimand_name}",
    )


In [ ]:
# Balance path plot. ATE and ATT are displayed separately.
plot_df = zhao_path_summary[zhao_path_summary["estimator"] == "rw"].copy()
for estimand_name in ESTIMANDS:
    panel_df = plot_df[plot_df["estimand"] == estimand_name].copy()
    if panel_df.empty:
        print(f"No path data for {estimand_name}.")
        continue
    fig, ax = plt.subplots(figsize=PLOT_CONFIG["figure_size"], dpi=PLOT_CONFIG["dpi"])
    for loss, g in panel_df.groupby("loss"):
        g = g.sort_values("n_active_features")
        ax.plot(g["n_active_features"], g["max_abs_smd_weighted_mean"], marker="o", linewidth=PLOT_CONFIG["line_width"], markersize=PLOT_CONFIG["marker_size"], color=METHOD_COLORS.get(loss), label=label_of(loss))
    ax.axhline(0.10, color="black", linestyle="--", linewidth=1.0, label="0.10 threshold")
    ax.set_title(f"{estimand_name}: Zhao/Kang-Schafer balance path", fontsize=PLOT_CONFIG["title_fontsize"])
    ax.set_xlabel("Number of active transformed features", fontsize=PLOT_CONFIG["axis_fontsize"])
    ax.set_ylabel("Mean max weighted SMD", fontsize=PLOT_CONFIG["axis_fontsize"])
    ax.grid(alpha=PLOT_CONFIG["grid_alpha"])
    ax.legend(fontsize=PLOT_CONFIG["legend_fontsize"])
    fig.tight_layout()
    plt.show()


## B. Kernel-basis mismatch

This experiment varies the outcome surface and compares polynomial, RKHS, and random Fourier bases under compatible loss-link pairs.

In [ ]:
rows = []
for outcome_kernel in KERNEL_OUTCOMES:
    for rep in range(N_REPLICATIONS):
        data = make_kernel_gp_data(n=SAMPLE_SIZE_KERNEL, d=6, seed=900000 + 1009 * rep, outcome_kernel=outcome_kernel)
        for basis_kind in KERNEL_BASIS_KINDS:
            for loss_spec in COMPATIBLE_LOSSES:
                for estimand in ESTIMANDS:
                    fit_rows = fit_one_grr(data, estimand=estimand, loss_spec=loss_spec, basis_kind=basis_kind, basis_mode="regressor", cross_fit=True, lam=RIEZ_LAMBDA, basis_features=120, folds=FOLDS, estimators=ESTIMATORS_ALL, random_state=rep)
                    for row in fit_rows:
                        row["outcome_surface"] = outcome_kernel
                        row["replication"] = rep
                    rows.extend(fit_rows)
kernel_results = pd.DataFrame(rows)
kernel_summary = summarize_estimates(kernel_results, ["outcome_surface", "estimand", "basis", "loss", "estimator"])

for estimand_name in ESTIMANDS:
    table_df = kernel_summary[kernel_summary["estimand"] == estimand_name].copy()
    if table_df.empty:
        print(f"No rows for {estimand_name}.")
        continue
    table_df = table_df.drop(columns=["estimand"])
    table_df = table_df.sort_values(["outcome_surface", "basis", "loss", "estimator"])
    display_table(table_df, caption=f"Kernel-basis mismatch summary: {estimand_name}")


In [ ]:
# Box plots for ARW squared error under basis mismatch. Separate by estimand and outcome surface.
plot_df = kernel_results[(kernel_results["status"] == "ok") & (kernel_results["estimator"] == "arw")].copy()
plot_df["method"] = plot_df["basis"] + " | " + plot_df["loss"]
plot_df["squared_error_plot"] = plot_df["squared_error"].clip(lower=PLOT_CONFIG["squared_error_floor"])

for estimand_name in ESTIMANDS:
    for outcome_kernel in KERNEL_OUTCOMES:
        panel_df = plot_df[(plot_df["estimand"] == estimand_name) & (plot_df["outcome_surface"] == outcome_kernel)].copy()
        if panel_df.empty:
            print(f"No plot data for {estimand_name}, {outcome_kernel}.")
            continue
        fig, ax = plt.subplots(figsize=PLOT_CONFIG["figure_size_wide"], dpi=PLOT_CONFIG["dpi"])
        ordered_methods = sorted(panel_df["method"].unique())
        box_data = [panel_df.loc[panel_df["method"] == m, "squared_error_plot"].dropna().to_numpy() for m in ordered_methods]
        box = ax.boxplot(box_data, tick_labels=[prettify_method(_m) for _m in ordered_methods], widths=PLOT_CONFIG["box_width"], patch_artist=True, showfliers=False)
        for patch, method in zip(box["boxes"], ordered_methods):
            basis_name = method.split(" | ")[0]
            patch.set_facecolor(METHOD_COLORS.get(basis_name, "#CCCCCC"))
            patch.set_alpha(0.75)
        ax.set_yscale(PLOT_CONFIG["squared_error_y_scale"])
        ax.set_title(f"{estimand_name}: kernel-basis mismatch, outcome={outcome_kernel}", fontsize=PLOT_CONFIG["title_fontsize"])
        ax.set_xlabel("Basis and loss", fontsize=PLOT_CONFIG["axis_fontsize"])
        ax.set_ylabel("Squared error", fontsize=PLOT_CONFIG["axis_fontsize"])
        ax.tick_params(axis="x", labelrotation=45, labelsize=PLOT_CONFIG["tick_fontsize"])
        ax.grid(axis="y", alpha=PLOT_CONFIG["grid_alpha"])
        fig.tight_layout()
        plt.show()
